In [2]:
import torch
from PIL import Image
from transformers import AutoModel, AutoProcessor
import torch
import open_clip
from PIL import Image
from pathlib import Path
from tqdm import tqdm
import io
import torch
from PIL import Image
from transformers import AutoProcessor, Siglip2VisionModel

device = "cuda" if torch.cuda.is_available() else "cpu"

/Users/georgianamg93gmail.com/interactions/bioclip2-env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/georgianamg93gmail.com/interactions/bioclip2-env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import os
import pandas as pd

def load_data(sys="/Volumes/NO NAME/images/", parquet_file = "../../data/benchmarks/taxa_co-occurence-interaction_benchmark.parquet"):

    # read as df
    df = pd.read_parquet(parquet_file,     
                        engine="pyarrow",
                        dtype_backend="pyarrow"
    )

    df["fileName"] = df["image_id"].astype(str) + df["file_ext"].astype(str)
    df["filePath"] = sys + df["fileName"].astype(str)

    # read all filenames present in folder
    files_in_folder = set(os.listdir(sys))

    # keep only rows where df["filename"] exists in folder
    df = df[df["fileName"].isin(files_in_folder)].copy()

    return df

In [4]:
import os
import pandas as pd

def load_data_all(sys="/Volumes/NO NAME/images/", parquet_file = "../../data/benchmarks/taxa_co-occurence-interaction_benchmark.parquet"):

    # read as df
    df = pd.read_parquet(parquet_file,     
                        engine="pyarrow",
                        dtype_backend="pyarrow"
    )

    df["fileName"] = df["image_id"].astype(str) + df["file_ext"].astype(str)
    df["filePath"] = sys + df["fileName"].astype(str)

    # read all filenames present in folder
    files_in_folder = set(os.listdir(sys))

    # keep only rows where df["filename"] exists in folder
    df = df[df["fileName"].isin(files_in_folder)].copy()

    return df

In [5]:
import os
import pandas as pd

def load_coralvqa_data(sys="/Volumes/NO NAME/images/", parquet_file = "../../data/benchmarks/taxa_co-occurence-interaction_benchmark.parquet"):

    # read as df
    df = pd.read_parquet(parquet_file,     
                        engine="pyarrow",
                        dtype_backend="pyarrow"
    )

    df["fileName"] = df["image_id"].astype(str)
    df["filePath"] = sys + df["fileName"].astype(str)

    # read all filenames present in folder
    files_in_folder = set(os.listdir(sys))

    # keep only rows where df["filename"] exists in folder
    df = df[df["fileName"].isin(files_in_folder)].copy()

    print(df["filePath"])

    return df

In [6]:
import os
import pandas as pd

def load_agmmu_data(sys="/Volumes/NO NAME/images/", parquet_file = "../../data/benchmarks/taxa_co-occurence-interaction_benchmark.parquet"):

    # read as df
    df = pd.read_parquet(parquet_file,     
                        engine="pyarrow",
                        dtype_backend="pyarrow"
    )

    # df["fileName"] = df["image_id"].astype(str)
    # df["filePath"] = sys + df["fileName"].astype(str)

    # # read all filenames present in folder
    # files_in_folder = set(os.listdir(sys))

    # # keep only rows where df["filename"] exists in folder
    # df = df[df["fileName"].isin(files_in_folder)].copy()

    # print(df["filePath"])

    print(df)

    return df

In [7]:
import os
import pandas as pd

def load_inquire_data(sys="/Volumes/NO NAME/images/", parquet_file = "../../data/benchmarks/taxa_co-occurence-interaction_benchmark.parquet"):

    # read as df
    df = pd.read_parquet(parquet_file,     
                        engine="pyarrow",
                        dtype_backend="pyarrow"
    )

    return df

#load_inquire_data("", "../data/benchmarks/inquire_interaction.parquet")

In [8]:
class SigLIP2Wrapper(torch.nn.Module):
    def __init__(self, model_name="google/siglip2-base-patch16-224", device="cpu"):
        super().__init__()
        self.model = AutoModel.from_pretrained(model_name)
        self.processor = AutoProcessor.from_pretrained(model_name)
        self.device = device

    def to(self, device):
        self.device = device
        self.model = self.model.to(device)
        return self

    def eval(self):
        self.model.eval()
        return self

    def encode_image(self, pixel_values):
        return self.model.get_image_features(pixel_values=pixel_values)

    def encode_text(self, input_ids=None, attention_mask=None, **kwargs):
        return self.model.get_text_features(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )


class SigLIP2Preprocess:
    def __init__(self, image_processor):
        self.image_processor = image_processor

    def __call__(self, image):
        out = self.image_processor(images=image, return_tensors="pt")
        return out["pixel_values"].squeeze(0)


class SigLIP2Tokenizer:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, texts):
        if isinstance(texts, str):
            texts = [texts]
        return self.tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=False,
        )

def create_siglip2_model_and_transforms(
    model_name="google/siglip2-base-patch16-224",
    device="cpu",
):
    model = SigLIP2Wrapper(model_name=model_name, device=device)
    preprocess = SigLIP2Preprocess(model.processor.image_processor)
    tokenizer = SigLIP2Tokenizer(model.processor.tokenizer)
    return model, preprocess, preprocess, tokenizer

In [13]:
def get_model_tokenizer(vlm):

    if vlm == "siglip2":
        # Load the SigLip2 as CLIP-like 
        model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
            "ViT-L-16-SigLIP2-256",
            pretrained="webli"
        )
        model = model.to(device).eval()
        tokenizer = open_clip.get_tokenizer("ViT-L-16-SigLIP2-256")

    if vlm == 'siglip':
        # Load the SigLip
        model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
            "ViT-SO400M-14-SigLIP",
            pretrained="webli",
        )
        model = model.to(device).eval()
        tokenizer = open_clip.get_tokenizer("ViT-SO400M-14-SigLIP")
    
    if vlm == 'bioclip2':
        # Load the BioCLIP 2 model from Hugging Face
        model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
            'hf-hub:imageomics/bioclip-2'
        )
        model = model.to(device).eval()
        tokenizer = open_clip.get_tokenizer('hf-hub:imageomics/bioclip-2')

    if vlm == "bioclip":
        # Load the BioCLIP model from Hugging Face
        model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
            'hf-hub:imageomics/bioclip'
        )
        model = model.to(device).eval()
        tokenizer = open_clip.get_tokenizer('hf-hub:imageomics/bioclip')
    
    if vlm == "clip":
        # Load the CLIP model from OpenAI
        model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
            'ViT-L-14',
            pretrained='openai'
        )
        model = model.to(device).eval()
        tokenizer = open_clip.get_tokenizer('ViT-L-14')

    if vlm == "metaclip":
        model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
            "ViT-L-14-quickgelu",
            pretrained="metaclip_fullcc"
        )
        model = model.to(device).eval()
        tokenizer = open_clip.get_tokenizer("ViT-L-14-quickgelu")

    if vlm == "taxabind":
        model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
            "hf-hub:MVRL/taxabind-vit-b-16"
        )
        model = model.to(device).eval()
        tokenizer = open_clip.get_tokenizer("hf-hub:MVRL/taxabind-vit-b-16")

    if vlm == "biocap":
        # Load the BioCAP model from Hugging Face
        model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
            "hf-hub:imageomics/biocap"
        )
        model = model.to(device).eval()
        tokenizer = open_clip.get_tokenizer("hf-hub:imageomics/biocap")

    if "biotrove" in vlm:
        if "biotrove-bioclip" in vlm:
            model_name = "hf-hub:imageomics/bioclip"
            ckpt_path = "biotrove-clip/biotroveclip-vit-b-16-from-bioclip-epoch-8.pt"
        if "biotrove-metaclip" in vlm:
            model_name = "ViT-L-14"
            ckpt_path = "biotrove-clip/biotroveclip-vit-l-14-from-metaclip-epoch-12.pt"
        if "biotrove-openai" in vlm:
            model_name = "ViT-B-16"
            ckpt_path = "biotrove-clip/biotroveclip-vit-b-16-from-openai-epoch-40.pt"

        model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
            model_name,
            pretrained=None,
        )
        tokenizer = open_clip.get_tokenizer(model_name)

        # Load checkpoint
        ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

        # Extract state dict if nested
        if isinstance(ckpt, dict):
            if "state_dict" in ckpt:
                state_dict = ckpt["state_dict"]
            elif "model_state_dict" in ckpt:
                state_dict = ckpt["model_state_dict"]
            else:
                state_dict = ckpt
        else:
            state_dict = ckpt

        # Remove common prefixes
        cleaned_state_dict = {}
        for k, v in state_dict.items():
            new_k = k
            if new_k.startswith("module."):
                new_k = new_k[len("module."):]
            if new_k.startswith("model."):
                new_k = new_k[len("model."):]
            cleaned_state_dict[new_k] = v

        missing, unexpected = model.load_state_dict(cleaned_state_dict, strict=False)

        print(f"Loaded checkpoint: {ckpt_path}")
        print(f"Missing keys: {len(missing)}")
        print(f"Unexpected keys: {len(unexpected)}")

        model = model.to(device).eval()

    return model, preprocess_val

for vlm in ["biotrove-bioclip","bioclip", "bioclip2", "siglip", "siglip2", "clip", "metaclip", "biocap", "taxabind", "biotrove-bioclip", "biotrove-openai", "biotrove-metaclip"]:
    
    model, preprocess_val = get_model_tokenizer(vlm)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"{vlm} Total params: {total_params:,}")


Loaded checkpoint: biotrove-clip/biotroveclip-vit-b-16-from-bioclip-epoch-8.pt
Missing keys: 0
Unexpected keys: 0
biotrove-bioclip Total params: 149,620,737
bioclip Total params: 149,620,737
bioclip2 Total params: 427,616,513
siglip Total params: 877,360,306
siglip2 Total params: 881,526,786
clip Total params: 427,616,513
metaclip Total params: 427,616,513
biocap Total params: 149,620,737
taxabind Total params: 149,620,737
Loaded checkpoint: biotrove-clip/biotroveclip-vit-b-16-from-bioclip-epoch-8.pt
Missing keys: 0
Unexpected keys: 0
biotrove-bioclip Total params: 149,620,737


Loaded checkpoint: biotrove-clip/biotroveclip-vit-b-16-from-openai-epoch-40.pt
Missing keys: 0
Unexpected keys: 0
biotrove-openai Total params: 149,620,737


Loaded checkpoint: biotrove-clip/biotroveclip-vit-l-14-from-metaclip-epoch-12.pt
Missing keys: 0
Unexpected keys: 0
biotrove-metaclip Total params: 427,616,513


In [ ]:
def get_image_embeddings(model, preprocess_val, batch_size, df):
    emb_list = []
    paths_ok = []
    image_paths = (df["filePath"]).tolist() 
    with torch.no_grad():
        for i in tqdm(range(0, len(image_paths), batch_size)):
            batch = image_paths[i:i+batch_size]

            imgs = []
            ok = []
            for p in batch:
                try:
                    img = preprocess_val(Image.open(p).convert("RGB"))
                    imgs.append(img)
                    ok.append(p)
                except Exception:
                    pass

            if not imgs:
                continue

            imgs = torch.stack(imgs).to(device)
            feats = model.encode_image(imgs)
            feats = feats / feats.norm(dim=-1, keepdim=True)  # normalize

            emb_list.append(feats.cpu())
            paths_ok.extend(ok)
            
        return emb_list,paths_ok
    

In [ ]:
def get_image_embeddings_bytes(model, preprocess_val, batch_size, df):
    emb_list = []
    idx_ok = []
    image_bytes = df["image"].tolist()

    with torch.no_grad():
        for i in tqdm(range(0, len(image_bytes), batch_size)):
            batch = image_bytes[i:i+batch_size]

            imgs = []
            ok_idx = []

            for j, b in enumerate(batch):
                try:
                    img = Image.open(io.BytesIO(b)).convert("RGB")
                    img = preprocess_val(img)
                    imgs.append(img)
                    ok_idx.append(i + j)
                except Exception:
                    pass

            if not imgs:
                continue

            imgs = torch.stack(imgs).to(device)
            feats = model.encode_image(imgs)
            feats = feats / feats.norm(dim=-1, keepdim=True)

            emb_list.append(feats.cpu())
            idx_ok.extend(ok_idx)
            
        return emb_list, idx_ok
    

In [ ]:
def save_embeddings(df, emb_list, paths_ok, folder, vlm, name):

    emb = torch.cat(emb_list, dim=0).numpy()  # (N, D)
    print("Embeddings:", emb.shape)

    subset = df[df["filePath"].isin(paths_ok)].reset_index(drop=True)

    torch.save({"emb": emb, "df": subset}, f"{folder}image_embeddings_{vlm}_{name}.pt")
    img_feat_data = torch.load(f"{folder}image_embeddings_{vlm}_{name}.pt", weights_only=False)
    

In [ ]:
def save_save_embeddings_bytes(df, emb_list, idx_ok, folder, vlm, name):
    emb = torch.cat(emb_list, dim=0).numpy()
    print("Embeddings:", emb.shape)

    subset = df.iloc[idx_ok].reset_index(drop=True)

    torch.save({"emb": emb, "df": subset}, f"{folder}image_embeddings_{vlm}_{name}.pt")
    img_feat_data = torch.load(f"{folder}image_embeddings_{vlm}_{name}.pt", weights_only=False)

# Run

In [ ]:
parquet_file = "../data/benchmarks/classification_balanced_benchmark.parquet" #"../data/benchmarks/classification_balanced_unique_benchmark.parquet" #local_path, e.g.: ../data/benchmarks/taxa_co-occurence-interaction_benchmark.parquet
sys = "/Volumes/Extreme SSD/interactions/inat_images/"#"/Volumes/Extreme SSD/interactions/inat_images/" # e.g.: "E:/images/" or "/mnt/e/images/" or "/Volumes/Extreme SDD/images/" or "/Volumes/NO NAME/images/" or "/Volumes/Extreme SSD/interactions/inat_images"
name = "biointeract40k_benchmark"#"biointeract40k_benchmark"
folder = "biointeract40k"#"data/benchmarks/biointeract40k/"  
df = load_data(sys=sys, parquet_file=parquet_file)
#df = load_coralvqa_data(sys=sys, parquet_file=parquet_file)
#df = load_agmmu_data(sys=sys, parquet_file=parquet_file)
#df = load_data_all(sys, parquet_file)

vlms=["bioclip", "bioclip2", "siglip", "siglip2", "clip", "metaclip", "biocap", "taxabind", "biotrove-bioclip", "biotrove-openai", "biotrove-metaclip"]

for vlm in vlms:

    model, preprocess_val = get_model_tokenizer(vlm)
    
    print("Using device:", device)

    # Get embeddings
    model = model.to(device).eval()

    batch_size = 32

    emb_list,paths_ok = get_image_embeddings(model, preprocess_val, batch_size, df)

    save_embeddings(df, emb_list, paths_ok, folder, vlm, name)


Inquire only

In [ ]:
parquet_file = "../data/benchmarks/inquire_interaction.parquet" #local_path, e.g.: ../data/benchmarks/taxa_co-occurence-interaction_benchmark.parquet
sys = "/Volumes/Extreme SSD/interactions/inat_images/" # e.g.: "E:/images/" or "/mnt/e/images/" or "/Volumes/Extreme SDD/images/" or "/Volumes/NO NAME/images/" or "/Volumes/Extreme SSD/interactions/inat_images"
name = "inquire_benchmark"
folder = "data/benchmarks/inquire/"  
df = load_inquire_data(sys="", parquet_file=parquet_file)

vlms=["bioclip", "bioclip2", "siglip", "siglip2", "clip", "metaclip", "biocap", "taxabind", "biotrove-bioclip", "biotrove-openai", "biotrove-metaclip"]
vlms = ["bioclip"]
for vlm in vlms:

    model, preprocess_val = get_model_tokenizer(vlm)
    
    print("Using device:", device)

    # Get embeddings
    model = model.to(device).eval()

    batch_size = 32

    emb_list,paths_ok = get_image_embeddings_bytes(model, preprocess_val, batch_size, df)

    save_save_embeddings_bytes(df, emb_list, paths_ok, folder, vlm, name)

# TEXT EMBEDDINGS

In [ ]:
# import re

# def split_camel(text):
#     if pd.isna(text):
#         return ""
#     return re.sub(r"([a-z])([A-Z])", r"\1 \2", str(text)).lower()

# subset["label"] = ( "an image of " +
#     subset["sourceTaxonName"].astype(str) + " and " +
#         #subset["interactionTypeName"].apply(split_camel) + " " +
#             subset["targetTaxonName"].astype(str)
#             )

# labels = subset["label"].unique().astype(str).tolist()   # column with texts / species names

# cols = ["sourceTaxonName", "targetTaxonName"]
# labels = pd.unique(subset[cols].values.ravel()).tolist()

# # subset = subset[
# #     (subset["sourceTaxonRank"] == "species") &
# #     (subset["targetTaxonRank"] == "species")
# # ]

# print(len(labels))
# labels

In [ ]:
# import torch
# import open_clip
# import pandas as pd
# from PIL import Image

# device = "cuda" if torch.cuda.is_available() else "cpu"

# # --- load BioCLIP-2 ---
# model, _, preprocess = open_clip.create_model_and_transforms("hf-hub:imageomics/bioclip-2")
# tokenizer = open_clip.get_tokenizer("hf-hub:imageomics/bioclip-2")
# model = model.to(device).eval()

# # ---------- encode labels in chunks ----------
# def encode_labels_in_chunks(labels, batch_size=512, keep_on_cpu=True):
#     feats_list = []
#     with torch.no_grad():
#         for i in range(0, len(labels), batch_size):
#             batch_labels = labels[i:i + batch_size]
#             text_tokens = tokenizer(batch_labels).to(device)

#             text_feats = model.encode_text(text_tokens)
#             text_feats = text_feats / text_feats.norm(dim=-1, keepdim=True)

#             feats_list.append(text_feats.cpu() if keep_on_cpu else text_feats)

#             # optional: reduce fragmentation / peak memory
#             del text_tokens, text_feats
#             if device == "cuda":
#                 torch.cuda.empty_cache()

#     text_features = torch.cat(feats_list, dim=0)
#     if not keep_on_cpu:
#         text_features = text_features.to(device)
#     return text_features

# # pick a safe batch size for 16GB; adjust if needed
# text_features = encode_labels_in_chunks(labels, batch_size=512, keep_on_cpu=True)

# print("Embeddings:", text_features.shape)

In [ ]:
# torch.save({"emb": text_features, "labels": labels}, "text_embeddings_unique.pt")
# data = torch.load("text_embeddings_unique.pt")